# 🌟 Treinamento de Modelos - LightGBM

## Objetivo
Este notebook foca no treinamento e otimização do modelo LightGBM para previsão de casos de dengue.

## Estratégia
- Utilizar os mesmos dados preparados
- Otimização específica para LightGBM com Optuna (se disponível) ou RandomizedSearch
- Comparação com Random Forest e XGBoost
- Análise de importância das features
- Early stopping para evitar overfitting

In [ ]:
# Importação das bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

import lightgbm as lgb
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import uniform, randint
import joblib
import warnings

warnings.filterwarnings('ignore')
plt.style.use('default')
sns.set_palette("husl")

print("✅ Bibliotecas carregadas com sucesso!")

In [ ]:
# Carregamento dos dados processados
print("📂 Carregando dados processados...")

df = pd.read_csv('dados_com_features.csv')
X = pd.read_csv('X_features.csv')
y = pd.read_csv('y_target.csv')['Quantidade_Casos']

# Recriar coluna de data para divisão temporal
df['Data'] = pd.to_datetime(df[['Ano', 'Mês']].assign(day=1))

print(f"📊 Dataset: {X.shape[0]:,} registros, {X.shape[1]} features")
print(f"🎯 Target: {y.shape[0]:,} valores")

In [ ]:
# Função de divisão temporal (mesma dos notebooks anteriores)
def split_time_series_data(df, X, y, train_end='2021-12', val_end='2023-12'):
    """
    Divide os dados de forma temporal para evitar data leakage
    """
    train_mask = df['Data'] <= pd.to_datetime(train_end)
    val_mask = (df['Data'] > pd.to_datetime(train_end)) & (df['Data'] <= pd.to_datetime(val_end))
    test_mask = df['Data'] > pd.to_datetime(val_end)

    X_train = X[train_mask]
    y_train = y[train_mask]
    X_val = X[val_mask]
    y_val = y[val_mask]
    X_test = X[test_mask]
    y_test = y[test_mask]

    return X_train, X_val, X_test, y_train, y_val, y_test, train_mask, val_mask, test_mask

# Função para calcular métricas
def calculate_metrics(y_true, y_pred, model_name="Modelo"):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100

    return {
        'Modelo': model_name,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R²': r2,
        'MAPE': mape
    }

# Dividir dados
X_train, X_val, X_test, y_train, y_val, y_test, train_mask, val_mask, test_mask = split_time_series_data(df, X, y)

print(f"📊 Divisão dos dados:")
print(f"   🏋️ Treino: {X_train.shape[0]:,} registros")
print(f"   🎯 Validação: {X_val.shape[0]:,} registros")
print(f"   🧪 Teste: {X_test.shape[0]:,} registros")

In [ ]:
# Converter dados para formato LightGBM Dataset
print("🔄 Convertendo dados para formato LightGBM...")

# Criar datasets LightGBM
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

print("✅ Datasets LightGBM criados!")

In [ ]:
# Modelo baseline LightGBM
print("🌟 Treinando modelo baseline LightGBM...")

# Parâmetros baseline
params_baseline = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'verbosity': -1,
    'seed': 42,
    'n_jobs': -1
}

# Treinamento com early stopping
lgb_baseline = lgb.train(
    params_baseline,
    train_data,
    num_boost_round=1000,
    valid_sets=[train_data, val_data],
    valid_names=['train', 'val'],
    early_stopping_rounds=50,
    verbose_eval=False
)

# Predições
y_pred_train_baseline = lgb_baseline.predict(X_train, num_iteration=lgb_baseline.best_iteration)
y_pred_val_baseline = lgb_baseline.predict(X_val, num_iteration=lgb_baseline.best_iteration)

# Métricas
metrics_train_baseline = calculate_metrics(y_train, y_pred_train_baseline, "LGB Baseline - Treino")
metrics_val_baseline = calculate_metrics(y_val, y_pred_val_baseline, "LGB Baseline - Validação")

print("📊 Métricas do modelo baseline:")
print("\n🏋️ Treino:")
for key, value in metrics_train_baseline.items():
    if key != 'Modelo':
        print(f"   {key}: {value:.4f}")

print("\n🎯 Validação:")
for key, value in metrics_val_baseline.items():
    if key != 'Modelo':
        print(f"   {key}: {value:.4f}")

print(f"\n🛑 Early stopping em: {lgb_baseline.best_iteration} iterações")

In [ ]:
# Otimização de hiperparâmetros com Optuna (se disponível) ou RandomizedSearch
def optimize_lgb_optuna():
    """Otimização com Optuna"""
    try:
        import optuna

        def objective(trial):
            params = {
                'objective': 'regression',
                'metric': 'rmse',
                'boosting_type': 'gbdt',
                'verbosity': -1,
                'seed': 42,
                'n_jobs': -1,
                'num_leaves': trial.suggest_int('num_leaves', 10, 300),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
                'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
                'bagging_fraction': trial.suggest_float('bagging_fraction', 0.4, 1.0),
                'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
                'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
                'reg_alpha': trial.suggest_float('reg_alpha', 0, 2),
                'reg_lambda': trial.suggest_float('reg_lambda', 0, 2),
            }

            # Combinação treino + validação para CV
            X_train_val = pd.concat([X_train, X_val])
            y_train_val = pd.concat([y_train, y_val])

            # Time Series Cross Validation
            tscv = TimeSeriesSplit(n_splits=3)
            scores = []

            for train_idx, val_idx in tscv.split(X_train_val):
                X_t, X_v = X_train_val.iloc[train_idx], X_train_val.iloc[val_idx]
                y_t, y_v = y_train_val.iloc[train_idx], y_train_val.iloc[val_idx]

                train_set = lgb.Dataset(X_t, label=y_t)
                val_set = lgb.Dataset(X_v, label=y_v, reference=train_set)

                model = lgb.train(
                    params,
                    train_set,
                    num_boost_round=1000,
                    valid_sets=[val_set],
                    early_stopping_rounds=50,
                    verbose_eval=False
                )

                pred = model.predict(X_v, num_iteration=model.best_iteration)
                rmse = np.sqrt(mean_squared_error(y_v, pred))
                scores.append(rmse)

            return np.mean(scores)

        study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
        study.optimize(objective, n_trials=30)

        return study.best_params, study.best_value

    except ImportError:
        return None, None

def optimize_lgb_random_search():
    """Otimização com RandomizedSearch"""
    print("🔧 Iniciando otimização com RandomizedSearch...")

    # Parâmetros para otimização
    param_distributions = {
        'num_leaves': randint(10, 300),
        'learning_rate': uniform(0.01, 0.29),
        'feature_fraction': uniform(0.4, 0.6),
        'bagging_fraction': uniform(0.4, 0.6),
        'bagging_freq': randint(1, 7),
        'min_child_samples': randint(5, 100),
        'reg_alpha': uniform(0, 2),
        'reg_lambda': uniform(0, 2),
        'n_estimators': randint(100, 1000)
    }

    # Cross-validation temporal
    tscv = TimeSeriesSplit(n_splits=3)

    # Modelo LightGBM para sklearn
    lgb_model = lgb.LGBMRegressor(
        objective='regression',
        boosting_type='gbdt',
        verbosity=-1,
        random_state=42,
        n_jobs=-1
    )

    # RandomizedSearch
    random_search = RandomizedSearchCV(
        lgb_model,
        param_distributions,
        n_iter=30,
        cv=tscv,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        verbose=1,
        random_state=42
    )

    # Fit na combinação treino + validação para otimização
    X_train_val = pd.concat([X_train, X_val])
    y_train_val = pd.concat([y_train, y_val])

    random_search.fit(X_train_val, y_train_val)

    return random_search.best_params_, -random_search.best_score_

# Tentar otimização com Optuna primeiro
print("🔧 Tentando otimização com Optuna...")
best_params_optuna, best_score_optuna = optimize_lgb_optuna()

if best_params_optuna is not None:
    print("✅ Otimização com Optuna concluída!")
    best_params = best_params_optuna
    best_score = best_score_optuna
    optimization_method = "Optuna"
else:
    print("⚠️ Optuna não disponível. Usando RandomizedSearch...")
    best_params, best_score = optimize_lgb_random_search()
    optimization_method = "RandomizedSearch"

print(f"\n🏆 Melhores hiperparâmetros ({optimization_method}):")
for param, value in best_params.items():
    print(f"   {param}: {value}")

print(f"\n📊 Melhor score (RMSE): {best_score:.4f}")

In [ ]:
# Modelo otimizado
print("🚀 Treinando modelo otimizado...")

# Parâmetros otimizados
params_optimized = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'verbosity': -1,
    'seed': 42,
    'n_jobs': -1
}

# Adicionar parâmetros otimizados (filtrando parâmetros específicos do sklearn se necessário)
sklearn_params = ['n_estimators']
for param, value in best_params.items():
    if param not in sklearn_params:
        params_optimized[param] = value

# Número de estimators
num_boost_round = best_params.get('n_estimators', 1000)

# Treinamento com early stopping
lgb_optimized = lgb.train(
    params_optimized,
    train_data,
    num_boost_round=num_boost_round,
    valid_sets=[train_data, val_data],
    valid_names=['train', 'val'],
    early_stopping_rounds=100,
    verbose_eval=False
)

# Predições
y_pred_train_opt = lgb_optimized.predict(X_train, num_iteration=lgb_optimized.best_iteration)
y_pred_val_opt = lgb_optimized.predict(X_val, num_iteration=lgb_optimized.best_iteration)
y_pred_test_opt = lgb_optimized.predict(X_test, num_iteration=lgb_optimized.best_iteration)

# Métricas
metrics_train_opt = calculate_metrics(y_train, y_pred_train_opt, "LGB Otimizado - Treino")
metrics_val_opt = calculate_metrics(y_val, y_pred_val_opt, "LGB Otimizado - Validação")
metrics_test_opt = calculate_metrics(y_test, y_pred_test_opt, "LGB Otimizado - Teste")

print("📊 Métricas do modelo otimizado:")
print("\n🏋️ Treino:")
for key, value in metrics_train_opt.items():
    if key != 'Modelo':
        print(f"   {key}: {value:.4f}")

print("\n🎯 Validação:")
for key, value in metrics_val_opt.items():
    if key != 'Modelo':
        print(f"   {key}: {value:.4f}")

print("\n🧪 Teste:")
for key, value in metrics_test_opt.items():
    if key != 'Modelo':
        print(f"   {key}: {value:.4f}")

# Comparação com baseline
print(f"\n📈 Melhoria na validação:")
print(f"   RMSE: {metrics_val_baseline['RMSE']:.4f} → {metrics_val_opt['RMSE']:.4f} ({((metrics_val_baseline['RMSE'] - metrics_val_opt['RMSE'])/metrics_val_baseline['RMSE']*100):+.2f}%)")
print(f"   R²: {metrics_val_baseline['R²']:.4f} → {metrics_val_opt['R²']:.4f} ({((metrics_val_opt['R²'] - metrics_val_baseline['R²'])/metrics_val_baseline['R²']*100):+.2f}%)")

print(f"\n🛑 Early stopping em: {lgb_optimized.best_iteration} iterações")

In [ ]:
# Curva de aprendizado
print("📈 Plotando curva de aprendizado...")

# Obter histórico do treinamento
train_results = lgb_optimized.eval_valid()

plt.figure(figsize=(12, 5))

# Loss durante o treinamento
plt.subplot(1, 2, 1)
iterations = range(1, len(train_results[0][2]) + 1)
plt.plot(iterations, train_results[0][2], label='Treino', linewidth=2)
plt.plot(iterations, train_results[1][2], label='Validação', linewidth=2)
plt.axvline(x=lgb_optimized.best_iteration, color='r', linestyle='--', label='Early Stop')
plt.xlabel('Iterações')
plt.ylabel('RMSE')
plt.title('Curva de Aprendizado - RMSE')
plt.legend()
plt.grid(True, alpha=0.3)

# Zoom na região do early stopping
plt.subplot(1, 2, 2)
start_idx = max(0, lgb_optimized.best_iteration - 200)
end_idx = min(len(iterations), lgb_optimized.best_iteration + 100)

zoom_iterations = iterations[start_idx:end_idx]
plt.plot(zoom_iterations, train_results[0][2][start_idx:end_idx],
         label='Treino', linewidth=2)
plt.plot(zoom_iterations, train_results[1][2][start_idx:end_idx],
         label='Validação', linewidth=2)
plt.axvline(x=lgb_optimized.best_iteration, color='r', linestyle='--', label='Early Stop')
plt.xlabel('Iterações')
plt.ylabel('RMSE')
plt.title('Zoom - Região Early Stopping')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Análise de importância das features
print("📊 Analisando importância das features...")

# Importância por ganho (gain)
importance_gain = lgb_optimized.feature_importance(importance_type='gain')
importance_split = lgb_optimized.feature_importance(importance_type='split')

# Criar DataFrame de importância
feature_names = X_train.columns.tolist()
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance_gain': importance_gain,
    'importance_split': importance_split
}).sort_values('importance_gain', ascending=False)

# Top 20 features mais importantes
top_features = importance_df.head(20)

fig, axes = plt.subplots(1, 2, figsize=(18, 10))

# Importância por ganho
sns.barplot(data=top_features, y='feature', x='importance_gain', ax=axes[0])
axes[0].set_title('Top 20 Features - Importância por Ganho')
axes[0].set_xlabel('Importância (Gain)')

# Importância por split
top_features_split = importance_df.head(20).sort_values('importance_split', ascending=False)
sns.barplot(data=top_features_split, y='feature', x='importance_split', ax=axes[1])
axes[1].set_title('Top 20 Features - Importância por Split')
axes[1].set_xlabel('Importância (Split)')

plt.tight_layout()
plt.show()

print("🏆 Top 10 features mais importantes (por ganho):")
for i, (_, row) in enumerate(top_features.head(10).iterrows(), 1):
    print(f"   {i:2d}. {row['feature']}: {row['importance_gain']:.2f}")

# Salvar importância das features
importance_df.to_csv('feature_importance_lgb.csv', index=False)
print("\n💾 Importância das features salva em 'feature_importance_lgb.csv'")

In [ ]:
# Análise por estado - dados de teste
print("🗺️ Analisando performance por estado...")

# Criar DataFrame com resultados do teste
df_test = df[test_mask].copy()
df_test['Predicoes'] = y_pred_test_opt
df_test['Residuos'] = df_test['Predicoes'] - df_test['Quantidade de Casos']

# Métricas por estado
metrics_by_state = []
for state in df_test['COD_UF'].unique():
    state_data = df_test[df_test['COD_UF'] == state]
    if len(state_data) > 0:
        metrics = calculate_metrics(
            state_data['Quantidade de Casos'],
            state_data['Predicoes'],
            state
        )
        metrics['Estado'] = state
        metrics['N_Obs'] = len(state_data)
        metrics_by_state.append(metrics)

metrics_df = pd.DataFrame(metrics_by_state)

print("\n🏆 Top 5 estados com melhor R²:")
for _, row in metrics_df.nlargest(5, 'R²').iterrows():
    print(f"   {row['Estado']}: R² = {row['R²']:.3f}, RMSE = {row['RMSE']:.2f}")

print("\n🎯 5 estados com menor R²:")
for _, row in metrics_df.nsmallest(5, 'R²').iterrows():
    print(f"   {row['Estado']}: R² = {row['R²']:.3f}, RMSE = {row['RMSE']:.2f}")

# Visualização das métricas por estado
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# R² por estado
axes[0, 0].bar(metrics_df['Estado'], metrics_df['R²'])
axes[0, 0].set_title('R² por Estado')
axes[0, 0].set_ylabel('R²')
axes[0, 0].tick_params(axis='x', rotation=45)

# RMSE por estado
axes[0, 1].bar(metrics_df['Estado'], metrics_df['RMSE'])
axes[0, 1].set_title('RMSE por Estado')
axes[0, 1].set_ylabel('RMSE')
axes[0, 1].tick_params(axis='x', rotation=45)

# MAE por estado
axes[1, 0].bar(metrics_df['Estado'], metrics_df['MAE'])
axes[1, 0].set_title('MAE por Estado')
axes[1, 0].set_ylabel('MAE')
axes[1, 0].tick_params(axis='x', rotation=45)

# MAPE por estado
axes[1, 1].bar(metrics_df['Estado'], metrics_df['MAPE'])
axes[1, 1].set_title('MAPE por Estado (%)')
axes[1, 1].set_ylabel('MAPE (%)')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Salvar métricas por estado
metrics_df.to_csv('metricas_por_estado_lgb.csv', index=False)
print("\n💾 Métricas por estado salvas em 'metricas_por_estado_lgb.csv'")

In [ ]:
# Visualização das predições
def plot_predictions(y_true, y_pred, title="Predições vs Real"):
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    # Scatter plot
    axes[0].scatter(y_true, y_pred, alpha=0.6)
    axes[0].plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=2)
    axes[0].set_xlabel('Valores Reais')
    axes[0].set_ylabel('Predições')
    axes[0].set_title(f'{title} - Scatter')

    # Residuos
    residuals = y_pred - y_true
    axes[1].scatter(y_pred, residuals, alpha=0.6)
    axes[1].axhline(y=0, color='r', linestyle='--')
    axes[1].set_xlabel('Predições')
    axes[1].set_ylabel('Resíduos')
    axes[1].set_title(f'{title} - Resíduos')

    plt.tight_layout()
    plt.show()

plot_predictions(y_test, y_pred_test_opt, "LightGBM Otimizado - Teste")

# Série temporal para alguns estados
estados_exemplo = ['SP', 'MG', 'RJ', 'BA', 'PR']

fig, axes = plt.subplots(len(estados_exemplo), 1, figsize=(15, 3*len(estados_exemplo)))

for i, estado in enumerate(estados_exemplo):
    state_data = df_test[df_test['COD_UF'] == estado].sort_values('Data')

    if len(state_data) > 0:
        axes[i].plot(state_data['Data'], state_data['Quantidade de Casos'],
                    label='Real', marker='o', linewidth=2)
        axes[i].plot(state_data['Data'], state_data['Predicoes'],
                    label='Predição', marker='s', linewidth=2, alpha=0.8)
        axes[i].set_title(f'Predições vs Real - {estado}')
        axes[i].set_ylabel('Casos de Dengue')
        axes[i].legend()
        axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Comparação com outros modelos (se disponíveis)
def compare_models():
    models_results = {}

    # LightGBM
    models_results['LightGBM'] = metrics_test_opt

    # Tentar carregar Random Forest
    try:
        rf_metrics = pd.read_csv('metricas_comparacao_rf.csv')
        rf_test = rf_metrics[rf_metrics['Modelo'].str.contains('Teste')]
        if len(rf_test) > 0:
            models_results['Random Forest'] = rf_test.iloc[0].to_dict()
    except FileNotFoundError:
        pass

    # Tentar carregar XGBoost
    try:
        xgb_metrics = pd.read_csv('metricas_comparacao_xgb.csv')
        xgb_test = xgb_metrics[xgb_metrics['Modelo'].str.contains('Teste')]
        if len(xgb_test) > 0:
            models_results['XGBoost'] = xgb_test.iloc[0].to_dict()
    except FileNotFoundError:
        pass

    if len(models_results) > 1:
        print("📊 COMPARAÇÃO ENTRE MODELOS")
        print("="*60)

        # Criar DataFrame para comparação
        comparison_df = pd.DataFrame(models_results).T

        # Selecionar métricas importantes
        metrics_to_compare = ['RMSE', 'MAE', 'R²', 'MAPE']

        print(f"\n🎯 Métricas no conjunto de TESTE:")
        print(f"{'Modelo':<15} {'RMSE':<10} {'MAE':<10} {'R²':<10} {'MAPE':<10}")
        print("-" * 60)

        for model_name, metrics in models_results.items():
            print(f"{model_name:<15} {metrics['RMSE']:<10.2f} {metrics['MAE']:<10.2f} {metrics['R²']:<10.4f} {metrics['MAPE']:<10.2f}")

        # Encontrar melhor modelo por métrica
        print(f"\n🏆 Melhor modelo por métrica:")
        best_rmse = min(models_results.items(), key=lambda x: x[1]['RMSE'])
        best_r2 = max(models_results.items(), key=lambda x: x[1]['R²'])
        print(f"   RMSE: {best_rmse[0]} ({best_rmse[1]['RMSE']:.4f})")
        print(f"   R²: {best_r2[0]} ({best_r2[1]['R²']:.4f})")

        # Visualização comparativa
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))

        metrics_plot = ['RMSE', 'MAE', 'R²', 'MAPE']
        for i, metric in enumerate(metrics_plot):
            row, col = i // 2, i % 2
            values = [models_results[model][metric] for model in models_results.keys()]
            axes[row, col].bar(models_results.keys(), values)
            axes[row, col].set_title(f'Comparação - {metric}')
            axes[row, col].set_ylabel(metric)
            axes[row, col].tick_params(axis='x', rotation=45)

        plt.tight_layout()
        plt.show()

        # Salvar comparação
        comparison_df.to_csv('comparacao_todos_modelos.csv')
        print("\n💾 Comparação salva em 'comparacao_todos_modelos.csv'")

    return models_results

models_comparison = compare_models()

In [ ]:
# Salvar modelo e resultados
print("💾 Salvando modelo treinado...")

# Salvar o modelo LightGBM
lgb_optimized.save_model('modelo_lightgbm.txt')
joblib.dump(lgb_optimized, 'modelo_lightgbm.pkl')
print("✅ Modelo salvo: 'modelo_lightgbm.pkl' e 'modelo_lightgbm.txt'")

# Salvar hiperparâmetros
with open('hiperparametros_lgb.txt', 'w', encoding='utf-8') as f:
    f.write("HIPERPARÂMETROS OTIMIZADOS - LIGHTGBM\n")
    f.write("="*50 + "\n\n")
    f.write(f"Método de otimização: {optimization_method}\n\n")
    for param, value in best_params.items():
        f.write(f"{param}: {value}\n")
    f.write(f"\nbest_iteration: {lgb_optimized.best_iteration}\n")
    f.write(f"total_boost_rounds: {num_boost_round}\n")

print("✅ Hiperparâmetros salvos: 'hiperparametros_lgb.txt'")

# Compilar todas as métricas
all_metrics = [
    metrics_train_baseline, metrics_val_baseline,
    metrics_train_opt, metrics_val_opt, metrics_test_opt
]

metrics_comparison = pd.DataFrame(all_metrics)
metrics_comparison.to_csv('metricas_comparacao_lgb.csv', index=False)
print("✅ Comparação de métricas salva: 'metricas_comparacao_lgb.csv'")

# Salvar predições do teste
test_predictions = pd.DataFrame({
    'Real': y_test,
    'Predicao': y_pred_test_opt,
    'Residuo': y_pred_test_opt - y_test
})
test_predictions.to_csv('predicoes_teste_lgb.csv', index=False)
print("✅ Predições do teste salvas: 'predicoes_teste_lgb.csv'")

In [ ]:
# Resumo final
print("🎯 RESUMO - LIGHTGBM")
print("="*50)
print(f"\n📊 Modelo Final:")
print(f"   Algoritmo: LightGBM Regressor")
print(f"   Método de otimização: {optimization_method}")
for param, value in best_params.items():
    print(f"   {param}: {value}")
print(f"   best_iteration: {lgb_optimized.best_iteration}")
print(f"   total_boost_rounds: {num_boost_round}")

print(f"\n📈 Performance (Teste):")
print(f"   RMSE: {metrics_test_opt['RMSE']:.2f}")
print(f"   MAE: {metrics_test_opt['MAE']:.2f}")
print(f"   R²: {metrics_test_opt['R²']:.4f}")
print(f"   MAPE: {metrics_test_opt['MAPE']:.2f}%")

print(f"\n🏆 Top 3 Features Mais Importantes:")
for i, (_, row) in enumerate(top_features.head(3).iterrows(), 1):
    print(f"   {i}. {row['feature']}: {row['importance_gain']:.2f}")

print(f"\n🗺️ Performance por Região:")
print(f"   Melhor estado (R²): {metrics_df.loc[metrics_df['R²'].idxmax(), 'Estado']} ({metrics_df['R²'].max():.3f})")
print(f"   Pior estado (R²): {metrics_df.loc[metrics_df['R²'].idxmin(), 'Estado']} ({metrics_df['R²'].min():.3f})")
print(f"   R² médio: {metrics_df['R²'].mean():.3f} ± {metrics_df['R²'].std():.3f}")

print(f"\n🛑 Early Stopping:")
print(f"   Melhor iteração: {lgb_optimized.best_iteration}")
print(f"   Total boost rounds: {num_boost_round}")

# Comparação final se houver outros modelos
if len(models_comparison) > 1:
    lgb_rank = sorted(models_comparison.items(), key=lambda x: x[1]['R²'], reverse=True)
    lgb_position = next(i for i, (name, _) in enumerate(lgb_rank, 1) if name == 'LightGBM')
    print(f"\n🥇 Ranking geral (por R²): {lgb_position}° lugar de {len(models_comparison)} modelos")

print("\n✅ LightGBM - Treinamento Concluído!")
print("Próximo: Comparação final e ensemble")